In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [ ]:
# @title Psyre imports

import logging
# !pip install -q psyre

# pylint: disable=g-importing-member,g-multiple-import
from psyre.builder import build_graph
from psyre.common import logger
from examples.rnn import (
    DEFAULT_MAX_STEPS,
    DIMENSION,
    Mode,
    ModelParams,
    PerceptronParams,
    RnnParams,
    TrainingState,
    extract_params,
    forward,
    hierarchical_model,
)
from psyre.scheduler import Scheduler
from psyre.stream import Stream
from psyre.trace import (
    is_present,
    present_values,
)
from psyre.visualize import visualize_graph

# pylint: enable=g-importing-member,g-multiple-import

In [ ]:
# @title Psyre model inference

logger.setLevel(logging.WARNING)
samples = [
    ([1.0, 2.0, -1.0, 0.0, 1.0], 2.0),
    ([0.5, -0.5, 1.0, 0.5, -1.0], 1.5),
]
params = ModelParams(
    rnn=RnnParams(wxh=0.2, whh=0.1, bh=0.0),
    per=PerceptronParams(wxh=[0.3] * DIMENSION, bh=[0.1] * DIMENSION),
)

horizon = (1 + len(samples)) * DIMENSION
_, _, _, eval_loss = Scheduler.run(
    hierarchical_model,
    dataset=samples,
    params=params,
    initial_mode=Mode.INFERENCE,
    horizon=horizon,
)
g_infer = Scheduler.graph
present_ticks = [idx for idx in range(horizon) if is_present(eval_loss, idx)]
eval_losses = present_values(eval_loss)
print('Present evaluation ticks:', present_ticks)
print('Evaluation losses:', eval_losses)

In [ ]:
# @title Psyre model training (DIMENSION=5, DEFAULT_MAX_STEPS=3)

train_horizon = (1 + DEFAULT_MAX_STEPS) * DIMENSION
_, param_traces, train_loss, _ = Scheduler.run(
    hierarchical_model,
    dataset=samples,
    initial_mode=Mode.TRAINING,
    max_steps=DEFAULT_MAX_STEPS,
    holdout_index=0,
    horizon=train_horizon,
)
g_train = Scheduler.graph
train_ticks = [
    idx for idx in range(train_horizon) if is_present(train_loss, idx)
]
train_losses = present_values(train_loss)
trained_params = extract_params(param_traces)
print('Present training ticks:', train_ticks)
print('Training losses:', train_losses)
print('Trained RNN params:', trained_params.rnn)

In [ ]:
# @title Stage-1 schematic: 2-layer RNN + perceptron forward pass (DIMENSION=5)


def forward_circuit() -> Stream:
  """Constructs the 2-layer RNN + ReLU perceptron forward circuit."""
  x = Stream(samples[0][0], name='x')
  param_streams = ModelParams(
      rnn=RnnParams(
          wxh=Stream(params.rnn.wxh, name='rnn.wxh'),
          whh=Stream(params.rnn.whh, name='rnn.whh'),
          bh=Stream(params.rnn.bh, name='rnn.bh'),
      ),
      per=PerceptronParams(
          wxh=Stream(params.per.wxh, name='per.wxh'),
          bh=Stream(params.per.bh, name='per.bh'),
      ),
  )
  return forward(x, param_streams)


g_forward = build_graph(forward_circuit)
visualize_graph(
    g_forward,
    title='Stage-1 SYNC Forward Pass (DIMENSION=5)',
    height_px=560,
)

In [ ]:
# @title Stage-1 schematic: hierarchical inference machine (DIMENSION=5)

visualize_graph(
    g_infer,
    title='Stage-1 SYNC Hierarchical Model (Mode.INFERENCE, DIMENSION=5)',
    height_px=640,
)

In [ ]:
# @title Stage-1 schematic: full training mode (DIMENSION=5, DEFAULT_MAX_STEPS=3)

visualize_graph(
    g_train,
    title=(
        'Stage-1 SYNC Hierarchical Model '
        '(Mode.TRAINING, DIMENSION=5, DEFAULT_MAX_STEPS=3)'
    ),
    height_px=680,
)

In [ ]:
# @title Stage-1 schematic: full hierarchical model (training + inference)


def full_hierarchical_circuit():
  """Constructs the two-mode training and inference hierarchical machine."""
  switch = Stream.once((1 + len(samples)) * DIMENSION - 1, name='switch_mode')
  return hierarchical_model(
      dataset=samples,
      params=params,
      initial_mode=Mode.INFERENCE,
      switch_mode=switch,
      train_init_state=TrainingState.INIT_LOAD,
      max_steps=DEFAULT_MAX_STEPS,
  )


g_hierarchical = build_graph(full_hierarchical_circuit)
visualize_graph(
    g_hierarchical,
    title='Stage-1 SYNC Full Hierarchical Model (Inference -> Training)',
    height_px=680,
)